In [0]:
%sql --name sql
--Inspecting Dataset
SELECT * 
FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`;

--checking for missing values
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN `Date` IS NULL THEN 1 ELSE 0 END) AS missing_dates,
    SUM(CASE WHEN `Sales` IS NULL THEN 1 ELSE 0 END) AS missing_sales,
    SUM(CASE WHEN `Cost of Sales` IS NULL THEN 1 ELSE 0 END) AS missing_cost_of_sales,
    SUM(CASE WHEN `Quantity Sold` IS NULL THEN 1 ELSE 0 END) AS missing_quantity
FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`;

--Checking for daily sales price per unit
SELECT
    `Date`,
    `Sales`,
    `Quantity Sold`,
    ROUND(`Sales` / NULLIF(`Quantity Sold`, 0), 2) AS daily_sales_price_per_unit
FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`
ORDER BY `Date`;

--checking average unit sale price
SELECT
    ROUND(
        SUM(`Sales`) / NULLIF(SUM(`Quantity Sold`), 0),
        2
    ) AS average_unit_sales_price
FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`;

--checking daily gross profit
SELECT
    `Date`,
    `Sales`,
    `Cost of Sales`,
        ROUND(
        (`Sales` - `Cost of Sales`) / NULLIF(`Sales`, 0) * 100,
        2
    ) AS daily_gross_profit_percentage
FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`
ORDER BY `Date`;

--putting gross profit in rands
SELECT
    `Date`,
    `Sales`,
    `Cost of Sales`,
        ROUND(
        `Sales` - `Cost of Sales`,
        2
    ) AS gross_profit_rand,
        ROUND(
        (`Sales` - `Cost of Sales`) / NULLIF(`Sales`, 0) * 100,
        2
    ) AS gross_profit_percentage
FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`
ORDER BY `Date`;

--checking daily gross profit per unit
SELECT
    `Date`,
        ROUND(
        (`Sales` - `Cost of Sales`) / NULLIF(`Quantity Sold`, 0),
        2
    ) AS gross_profit_per_unit
FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`
ORDER BY `Date`;

--checking gross profit per unit percentage
WITH per_unit AS (
  SELECT
    `Date`,
    `Sales` / NULLIF(`Quantity Sold`, 0) AS sales_price_per_unit_raw,
    (`Sales` - `Cost of Sales`) / NULLIF(`Quantity Sold`, 0) AS gross_profit_per_unit_raw
  FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`
)
SELECT
  `Date`,
  ROUND(sales_price_per_unit_raw, 2) AS sales_price_per_unit,
  ROUND(gross_profit_per_unit_raw, 2) AS gross_profit_per_unit,
  ROUND(
    gross_profit_per_unit_raw
    / NULLIF(sales_price_per_unit_raw, 0) * 100,
    2
  ) AS gross_profit_per_unit_percentage
FROM per_unit
ORDER BY `Date`;

--creating a table
CREATE OR REPLACE TEMP VIEW sales_analysis AS
SELECT
    `Date`,
    `Sales`,
    `Cost of Sales`,
    `Quantity Sold`,
        -- Unit Selling Price
    `Sales` / NULLIF(`Quantity Sold`, 0) AS unit_sales_price,
        -- Gross Profit
    `Sales` - `Cost of Sales` AS gross_profit,
        -- Gross Profit Percentage
    (`Sales` - `Cost of Sales`) /
        NULLIF(`Sales`, 0) * 100 AS gross_profit_percentage,
        -- Gross Profit Per Unit
    (`Sales` - `Cost of Sales`) /
        NULLIF(`Quantity Sold`, 0) AS gross_profit_per_unit
FROM `fnb-sales-data`.`fnb-sales-dataset`.`fnb-sales-data-code`;

SELECT *
FROM sales_analysis
ORDER BY `Date`;

--checking daily sales and quantity
SELECT
    `Date`,
    ROUND(unit_sales_price, 2) AS unit_price,
    `Quantity Sold`,
    `Sales`
FROM sales_analysis
ORDER BY unit_sales_price ASC;

--checking for promotion days
SELECT
    `Date`,
    ROUND(unit_sales_price, 2) AS unit_price,
    `Quantity Sold`,
        CASE
        WHEN unit_sales_price <
             (SELECT AVG(unit_sales_price) * 0.90
              FROM sales_analysis)
        THEN 'Potential Promotion'
        ELSE 'Regular Price'
    END AS promotion_flag
FROM sales_analysis
ORDER BY `Date`;

--checking previous days values
SELECT
    `Date`,
    unit_sales_price,
    `Quantity Sold`,
        LAG(unit_sales_price) OVER (ORDER BY `Date`) AS previous_price,
        LAG(`Quantity Sold`) OVER (ORDER BY `Date`) AS previous_quantity
FROM sales_analysis
ORDER BY `Date`;

--checking for promotion periods
SELECT
    `Date`,
    ROUND(unit_sales_price, 2) AS price,
    `Quantity Sold`,
    gross_profit,
    gross_profit_percentage
FROM sales_analysis
WHERE `Date` BETWEEN '2025-01-01' AND '2025-01-31'
ORDER BY `Date`;

--checking for promotion performance
SELECT
  promotion_flag,
  ROUND(AVG(unit_sales_price), 2) AS avg_price,
  ROUND(AVG(`Quantity Sold`), 0) AS avg_daily_quantity,
  ROUND(SUM(`Sales`), 2) AS total_sales,
  ROUND(SUM(gross_profit), 2) AS total_gross_profit,
  ROUND(SUM(gross_profit) * 100.0 / NULLIF(SUM(`Sales`), 0), 2) AS gross_profit_margin
FROM (
  SELECT
    *,
    CASE
      WHEN unit_sales_price < (SELECT AVG(unit_sales_price) * 0.90 FROM sales_analysis)
        THEN 'Potential Promotion'
      ELSE 'Regular Price'
    END AS promotion_flag
  FROM sales_analysis
)
GROUP BY promotion_flag;

--checking Total Sales, Cost and Profit
SELECT
    ROUND(SUM(`Sales`), 2) AS total_sales,
    ROUND(SUM(`Cost of Sales`), 2) AS total_cost_of_sales,
    ROUND(SUM(`Sales`) - SUM(`Cost of Sales`), 2) AS total_gross_profit,
    ROUND(
        (SUM(`Sales`) - SUM(`Cost of Sales`)) /
        NULLIF(SUM(`Sales`), 0) * 100,
        2
    ) AS overall_gross_profit_margin
FROM sales_analysis;

--checking best sales day
SELECT
    `Date`,
    `Sales`,
    `Quantity Sold`,
    gross_profit
FROM sales_analysis
ORDER BY `Sales` DESC
LIMIT 10;

--checking most profitable days
SELECT
    `Date`,
    gross_profit,
    gross_profit_percentage,
    `Sales`
FROM sales_analysis
ORDER BY gross_profit DESC
LIMIT 10;

SELECT
    `Date`,
    ROUND(unit_sales_price, 2) AS unit_price,
    `Quantity Sold`
FROM sales_analysis
ORDER BY unit_price ASC
LIMIT 10;

SELECT
    `Date`,
    ROUND(unit_sales_price, 2) AS unit_price,
    `Quantity Sold`
FROM sales_analysis
ORDER BY `Quantity Sold` DESC
LIMIT 10;


SELECT
    `Date`,
    ROUND(unit_sales_price, 2) AS unit_price,
    `Quantity Sold`
FROM sales_analysis
ORDER BY `Quantity Sold` DESC
;